In [1]:
taxi_year   = 2024
write_mode  = "overwrite"

StatementMeta(, , -1, SessionStarting, , SessionStarting, True)

In [ ]:
import json
from datetime import datetime, timezone

def utcnow() -> str:
    return datetime.now(timezone.utc).isoformat()

run_log = {
    "run_id":  datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"),
    "phase":   "gold",
    "started": utcnow(),
    "steps":   {},
}

def run_step(name: str, notebook: str, args: dict = None):
    print(f"\n{'─'*60}")
    print(f"  Running : {notebook}")
    print(f"{'─'*60}\n")
    try:
        notebookutils.notebook.run(
            notebook,
            timeout_seconds=7200,
            arguments=args or {},
        )
        run_log["steps"][name] = {"status": "success", "ts": utcnow()}
        print(f"\n  [PASS] {name}")
    except Exception as exc:
        run_log["steps"][name] = {"status": "failed", "error": str(exc), "ts": utcnow()}
        print(f"\n  [FAIL] {name}: {exc}")

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

# Cast injected parameters to correct types
taxi_year  = int(taxi_year)
write_mode = str(write_mode)

GOLD_BASE = (
    "abfss://itransition_de_project@onelake.dfs.fabric.microsoft.com"
    "/gold.Lakehouse"
)

REQUIRED = [
    "dimdate", "dimzone", "dimlocation", "dimfx", "dimgdp",
    "facttaxidaily", "factairqualitydaily",
]

print("Verifying gold lakehouse tables ...")
missing = []
for table in REQUIRED:
    path = f"{GOLD_BASE}/Tables/dbo/{table}"
    try:
        spark.read.format("delta").load(path).limit(0).count()
        print(f"  [OK]      {table}")
    except Exception:
        print(f"  [MISSING] {table}")
        missing.append(table)

if missing:
    raise RuntimeError(
        f"Missing gold tables: {missing}. "
        f"Run 02_gold_dim_load and fact notebooks first."
    )

print("\nAll required tables confirmed.\n")

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:

run_step(
    name     = "dim_tables",
    notebook = "02_gold_dim_load",
    args     = {"write_mode": write_mode},
)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
run_step(
    name     = "fact_taxi_daily",
    notebook = "03_gold_fact_taxi",
    args     = {"year": taxi_year, "write_mode": write_mode},
)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
run_step(
    name     = "fact_air_quality_daily",
    notebook = "04_gold_fact_airquality",
    args     = {"write_mode": write_mode},
)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
run_log["finished"] = utcnow()

successes = sum(1 for v in run_log["steps"].values() if v["status"] == "success")
failures  = sum(1 for v in run_log["steps"].values() if v["status"] == "failed")

print(f"\n{'='*60}")
print(f"  GOLD LOAD COMPLETE")
print(f"  Passed : {successes}/3 steps")
print(f"  Failed : {failures}/3 steps")
print(f"  Run ID : {run_log['run_id']}")
print(f"{'='*60}\n")
print(json.dumps(run_log, indent=2))

import os
log_dir = "/lakehouse/default/Files/_run_logs"
os.makedirs(log_dir, exist_ok=True)
log_path = os.path.join(log_dir, f"gold_run_{run_log['run_id']}.json")
with open(log_path, "w") as f:
    json.dump(run_log, f, indent=2)
print(f"\n  [OK] Run log saved: {log_path}")

if failures > 0:
    raise RuntimeError(
        f"Gold load completed with {failures} failure(s). "
        f"Check run log: {log_path}"
    )

StatementMeta(, , -1, Waiting, , Waiting, True)